In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQ_G.xpt"
csv_file = DATA_DIR / "SMQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 SMQ_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMQ020",
    "SMD030",
    "SMQ040",
    "SMQ050Q",
    "SMQ050U",
    "SMD055",
    "SMD057",
    "SMQ077",
    "SMD641",
    "SMD650",
    "SMD093",
    "SMDUPCA",
    "SMD100BR",
    "SMD100FL",
    "SMD100MN",
    "SMD100LN",
    "SMD100TR",
    "SMD100NI",
    "SMD100CO",
    "SMQ621",
    "SMD630",
    "SMQ660",
    "SMQ664M",
    "SMQ664C",
    "SMQ664W",
    "SMQ664B",
    "SMQ664O",
    "SMQ670",
    "SMAQUEX2"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 6790:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 30:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,790 rows x 30 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# CRITICAL:
#
# SMD100NI = FTC nicotine content.
#
# This is a GENUINE DECIMAL field.
#
# Example released values:
#
#     0.8
#     0.9
#     1.0
#     1.1
#     1.6
#
# DO NOT:
#
#     round SMD100NI
#     convert it to Int64
#     globally remove ".0"
#
#
# SMDUPCA and SMD100BR are text fields.
#
# SMDUPCA is especially important because many UPCs
# begin with zero.
#

DECIMAL_COLUMNS = [
    "SMD100NI"
]


STRING_COLUMNS = [
    "SMDUPCA",
    "SMD100BR"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if (
        col not in DECIMAL_COLUMNS
        and
        col not in STRING_COLUMNS
    )
]


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly one genuine decimal field."
    )


if len(STRING_COLUMNS) != 2:

    raise ValueError(
        "Expected exactly two string fields."
    )


if len(INTEGER_COLUMNS) != 27:

    raise ValueError(
        "Expected exactly 27 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK ORIGINAL STRING FIELD TYPES
# ============================================================

#
# Do NOT allow UPC to become numeric.
#

for col in STRING_COLUMNS:

    if pd.api.types.is_numeric_dtype(
        df[col]
    ):

        raise ValueError(
            f"{col} unexpectedly loaded as numeric."
        )


print(
    "PASS: UPC and cigarette-brand fields "
    "loaded as text/object."
)


# ============================================================
# 6. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER replace this globally before validating
# the exact variables and counts.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached SMQ_G.xpt pattern:
#
# SMD030   =  58
# SMD641   =  89
# SMD100FL =  13
# SMD100MN = 652
#
# TOTAL = 812
#
#
# CDC confirms:
#
# SMD030:
#     0 = Never smoked cigarettes regularly
#
# SMD641:
#     valid range includes 0-30 days
#
# SMD100FL:
#     0 = Non-filter
#
# SMD100MN:
#     0 = Non-menthol
#

EXPECTED_TINY_COUNTS = {
    "SMD030": 58,
    "SMD641": 89,
    "SMD100FL": 13,
    "SMD100MN": 652
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated SMQ_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 812:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "existed before correction."
    )


print(
    "\nPASS: Exact 812-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 8. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Restore only the four proven variables.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 812:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 9. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# XPT should contain.
#
# ONLY validated tiny -> zero positions may change.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 10. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining after correction:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# Attached SMQ_G contains genuine fractional
# observations only in:
#
#     SMD100NI
#

print("\n--- GENUINE DECIMAL CHECK ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{fractional_count:,} "
            "fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern changed.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "\nPASS: SMD100NI is the only genuine "
    "decimal variable."
)


# ============================================================
# 12. VALIDATE SMD100NI FRACTIONAL VALUES
# ============================================================

nicotine_values = (
    df["SMD100NI"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


fractional_mask = ~np.isclose(
    nicotine_values,
    np.round(nicotine_values),
    rtol=0,
    atol=1e-12
)


nicotine_fractional_count = int(
    fractional_mask.sum()
)


nicotine_whole_count = int(
    (
        ~fractional_mask
    ).sum()
)


print("\n--- SMD100NI ---")


print(
    "Non-missing values:",
    len(nicotine_values)
)


print(
    "Fractional observations:",
    nicotine_fractional_count
)


print(
    "Whole-looking decimal observations:",
    nicotine_whole_count
)


print(
    "Minimum:",
    np.min(
        nicotine_values
    )
)


print(
    "Maximum:",
    np.max(
        nicotine_values
    )
)


if nicotine_fractional_count != 767:

    raise ValueError(
        "Unexpected SMD100NI fractional count."
    )


if nicotine_whole_count != 58:

    raise ValueError(
        "Unexpected SMD100NI whole-looking count."
    )


if not (
    np.isclose(
        np.min(nicotine_values),
        0.1,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        np.max(nicotine_values),
        1.9,
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "Unexpected SMD100NI range."
    )


print(
    "PASS: SMD100NI genuine decimals validated."
)


# ============================================================
# 13. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 14. FORMAT ONLY PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# Convert only the 27 proven whole-number numeric fields.
#
# EXCLUDE:
#
#     SMD100NI  -> genuine decimal
#     SMDUPCA   -> text
#     SMD100BR  -> text
#
# NO .round() is used for conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine decimal values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 27 proven whole-number fields "
    "converted to nullable Int64."
)

print(
    "PASS: SMD100NI left as decimal."
)

print(
    "PASS: SMDUPCA and SMD100BR left as text."
)


# ============================================================
# 15. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} frequencies validated."
    )


# ============================================================
# 16. CDC CHECK - SMQ020
# ============================================================

#
# Smoked at least 100 cigarettes in life.
#

check_frequency(
    "SMQ020",
    {
        1: 2369,
        2: 3184,
        7: 2,
        9: 4
    },
    1231
)


# ============================================================
# 17. CDC CHECK - SMD030
# ============================================================

#
# Age started smoking cigarettes regularly.
#
# CDC:
#
# 6-72 = 2,292
# 0 Never = 58
# 777 Refused = 2
# 999 Don't know = 17
# Missing = 4,421
#

s = df["SMD030"]


if not (
    int(
        s.between(
            6,
            72
        ).sum()
    ) == 2292

    and int(
        s.eq(0).sum()
    ) == 58

    and int(
        s.eq(80).sum()
    ) == 0

    and int(
        s.eq(777).sum()
    ) == 2

    and int(
        s.eq(999).sum()
    ) == 17

    and int(
        s.isna().sum()
    ) == 4421
):

    raise ValueError(
        "SMD030 differs from validated CDC pattern."
    )


print(
    "PASS: SMD030 matches CDC including "
    "58 restored zeros."
)


# ============================================================
# 18. CDC CHECK - SMQ040
# ============================================================

check_frequency(
    "SMQ040",
    {
        1: 912,
        2: 196,
        3: 1259,
        7: 2
    },
    4421
)


# ============================================================
# 19. CDC CHECK - SMQ050Q / SMQ050U
# ============================================================

s = df["SMQ050Q"]


if not (
    int(
        s.between(
            1,
            193
        ).sum()
    ) == 1259

    and int(
        s.eq(77777).sum()
    ) == 1

    and int(
        s.eq(99999).sum()
    ) == 6

    and int(
        s.isna().sum()
    ) == 5524
):

    raise ValueError(
        "SMQ050Q differs from CDC."
    )


check_frequency(
    "SMQ050U",
    {
        1: 9,
        2: 11,
        3: 91,
        4: 1148
    },
    5531
)


print(
    "PASS: SMQ050Q / SMQ050U validated."
)


# ============================================================
# 20. CDC CHECK - SMD641
# ============================================================

#
# Number of days smoked cigarettes during past 30 days.
#
# CDC valid range:
#
#     0-30
#
# Attached file has:
#
#     zero = 89
#
# plus:
#
#     99 = Don't know = 3
#

s = df["SMD641"]


if not (
    int(
        s.between(
            0,
            30
        ).sum()
    ) == 1310

    and int(
        s.eq(0).sum()
    ) == 89

    and int(
        s.eq(77).sum()
    ) == 0

    and int(
        s.eq(99).sum()
    ) == 3

    and int(
        s.isna().sum()
    ) == 5477
):

    raise ValueError(
        "SMD641 differs from validated release pattern."
    )


print(
    "PASS: SMD641 matches CDC range and "
    "contains 89 restored zeros."
)


# ============================================================
# 21. CDC CHECK - SMD650
# ============================================================

s = df["SMD650"]


if not (
    int(
        s.between(
            1,
            95
        ).sum()
    ) == 1217

    and int(
        s.eq(999).sum()
    ) == 6

    and int(
        s.isna().sum()
    ) == 5567
):

    raise ValueError(
        "SMD650 differs from CDC."
    )


print(
    "PASS: SMD650 matches CDC."
)


# ============================================================
# 22. CDC CHECK - SMD093
# ============================================================

check_frequency(
    "SMD093",
    {
        1: 672,
        2: 388,
        3: 27,
        4: 20,
        7: 1
    },
    5682
)


# ============================================================
# 23. VALIDATE SMDUPCA AS TEXT
# ============================================================

#
# SMDUPCA is a 12-character UPC/string field.
#
# Attached file:
#
# recorded = 672
# blank    = 6118
#
# 585 recorded values begin with "0".
#
# Converting this field numerically would destroy
# those leading zeros.
#

upc = (
    df["SMDUPCA"]
    .where(
        df["SMDUPCA"].notna(),
        ""
    )
    .astype(str)
)


upc_nonblank = (
    upc != ""
)


upc_recorded_count = int(
    upc_nonblank.sum()
)


upc_blank_count = int(
    (
        ~upc_nonblank
    ).sum()
)


upc_length_12_count = int(
    upc[
        upc_nonblank
    ]
    .str.len()
    .eq(12)
    .sum()
)


upc_leading_zero_count = int(
    upc[
        upc_nonblank
    ]
    .str.startswith("0")
    .sum()
)


print("\n--- SMDUPCA ---")


print(
    "Recorded UPC values:",
    upc_recorded_count
)


print(
    "Blank UPC values:",
    upc_blank_count
)


print(
    "12-character recorded values:",
    upc_length_12_count
)


print(
    "Recorded values beginning with 0:",
    upc_leading_zero_count
)


if upc_recorded_count != 672:

    raise ValueError(
        "Unexpected SMDUPCA recorded count."
    )


if upc_blank_count != 6118:

    raise ValueError(
        "Unexpected SMDUPCA blank count."
    )


if upc_length_12_count != 672:

    raise ValueError(
        "SMDUPCA contains a recorded value "
        "that is not 12 characters."
    )


if upc_leading_zero_count != 585:

    raise ValueError(
        "Unexpected SMDUPCA leading-zero count."
    )


print(
    "PASS: SMDUPCA preserved as exact "
    "12-character text."
)


# ============================================================
# 24. VALIDATE SMD100BR AS TEXT
# ============================================================

brand = (
    df["SMD100BR"]
    .where(
        df["SMD100BR"].notna(),
        ""
    )
    .astype(str)
)


brand_recorded_count = int(
    brand.ne("")
    .sum()
)


brand_blank_count = int(
    brand.eq("")
    .sum()
)


print("\n--- SMD100BR ---")


print(
    "Recorded brand values:",
    brand_recorded_count
)


print(
    "Blank brand values:",
    brand_blank_count
)


if brand_recorded_count != 1026:

    raise ValueError(
        "Unexpected SMD100BR recorded count."
    )


if brand_blank_count != 5764:

    raise ValueError(
        "Unexpected SMD100BR blank count."
    )


print(
    "PASS: SMD100BR text field validated."
)


# ============================================================
# 25. CDC CHECK - SMD100FL
# ============================================================

#
# 0 = Non-filter
# 1 = Filter
#

check_frequency(
    "SMD100FL",
    {
        0: 13,
        1: 1009
    },
    5768
)


# ============================================================
# 26. CDC CHECK - SMD100MN
# ============================================================

#
# 0 = Non-menthol
# 1 = Menthol
#

check_frequency(
    "SMD100MN",
    {
        0: 652,
        1: 371
    },
    5767
)


# ============================================================
# 27. CDC CHECK - SMD100LN
# ============================================================

check_frequency(
    "SMD100LN",
    {
        1: 56,
        2: 542,
        3: 415,
        4: 11
    },
    5766
)


# ============================================================
# 28. PRODUCT CONTENT RANGE CHECKS
# ============================================================

#
# IMPORTANT:
#
# CDC webpage currently displays 966 nonmissing
# observations for SMD100TR, SMD100NI and SMD100CO.
#
# The attached XPT has 825 nonmissing observations
# for each of these three fields.
#
# Because this task is fidelity-preserving conversion,
# DO NOT create values for the additional records.
#
# Validate the attached values and preserve its exact
# missing-value pattern.
#

PRODUCT_CONTENT_COLUMNS = [
    "SMD100TR",
    "SMD100NI",
    "SMD100CO"
]


print("\n--- PRODUCT CONTENT CHECK ---")


for col in PRODUCT_CONTENT_COLUMNS:

    print(
        f"{col}: "
        f"{int(df[col].notna().sum()):,} "
        "non-missing in attached XPT"
    )


if int(
    df["SMD100TR"]
    .notna()
    .sum()
) != 825:

    raise ValueError(
        "Unexpected attached SMD100TR count."
    )


if int(
    df["SMD100NI"]
    .notna()
    .sum()
) != 825:

    raise ValueError(
        "Unexpected attached SMD100NI count."
    )


if int(
    df["SMD100CO"]
    .notna()
    .sum()
) != 825:

    raise ValueError(
        "Unexpected attached SMD100CO count."
    )


#
# All three content variables should have the
# same nonmissing positions in the attached XPT.
#

if not np.array_equal(
    df["SMD100TR"]
    .notna()
    .to_numpy(
        dtype=bool
    ),
    df["SMD100NI"]
    .notna()
    .to_numpy(
        dtype=bool
    )
):

    raise ValueError(
        "SMD100TR and SMD100NI missing patterns differ."
    )


if not np.array_equal(
    df["SMD100TR"]
    .notna()
    .to_numpy(
        dtype=bool
    ),
    df["SMD100CO"]
    .notna()
    .to_numpy(
        dtype=bool
    )
):

    raise ValueError(
        "SMD100TR and SMD100CO missing patterns differ."
    )


#
# CDC published ranges:
#
# Tar:
#     2-20
#
# Nicotine:
#     0.1-1.9
#
# Carbon monoxide:
#     3-20
#
# Attached values fall within those ranges.
#

if not (
    df["SMD100TR"]
    .dropna()
    .between(
        2,
        20
    )
    .all()
):

    raise ValueError(
        "SMD100TR contains value outside CDC range."
    )


if not (
    df["SMD100NI"]
    .dropna()
    .between(
        0.1,
        1.9
    )
    .all()
):

    raise ValueError(
        "SMD100NI contains value outside CDC range."
    )


if not (
    df["SMD100CO"]
    .dropna()
    .between(
        3,
        20
    )
    .all()
):

    raise ValueError(
        "SMD100CO contains value outside CDC range."
    )


print(
    "PASS: Cigarette product content values "
    "are within CDC released ranges."
)


# ============================================================
# 29. CDC CHECK - SMQ621
# ============================================================

#
# Youth lifetime cigarette count category.
#

check_frequency(
    "SMQ621",
    {
        1: 854,
        2: 121,
        3: 13,
        4: 29,
        5: 35,
        6: 30,
        7: 26,
        8: 66,
        99: 1
    },
    5615
)


# ============================================================
# 30. CDC CHECK - SMD630
# ============================================================

#
# Age first smoked a whole cigarette.
#
# Released valid values in attached XPT:
# 7-19 years.
#

s = df["SMD630"]


if not (
    int(
        s.between(
            7,
            19
        ).sum()
    ) == 198

    and int(
        s.isna().sum()
    ) == 6592
):

    raise ValueError(
        "SMD630 differs from CDC."
    )


print(
    "PASS: SMD630 matches CDC."
)


# ============================================================
# 31. CDC CHECK - SMQ660 / SMQ670
# ============================================================

check_frequency(
    "SMQ660",
    {
        1: 42,
        2: 23,
        3: 39,
        4: 4,
        5: 1,
        8: 13
    },
    6668
)


check_frequency(
    "SMQ670",
    {
        1: 71,
        2: 50,
        9: 1
    },
    6668
)


# ============================================================
# 32. CDC CHECK - QUESTIONNAIRE MODE
# ============================================================

#
# 1 = Home Interview, age 20+
# 2 = A-CASI, age 12-19
#

check_frequency(
    "SMAQUEX2",
    {
        1: 5560,
        2: 1230
    },
    0
)


# ============================================================
# 33. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Raw attached XPT contained no ordinary zeros.
#
# Therefore, after correction, every zero should occur
# only in the four validated variables.
#

final_zero_counts = {}


for col in numeric_cols:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from expected.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {final_zero_counts}"
    )


print(
    "PASS: Final numeric zero pattern exactly "
    "matches validated XPORT corrections."
)


# ============================================================
# 34. NUMERIC VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY permitted numeric differences from
# pandas-decoded source XPT are:
#
# SMD030:
#     58 tiny -> 0
#
# SMD641:
#     89 tiny -> 0
#
# SMD100FL:
#     13 tiny -> 0
#
# SMD100MN:
#     652 tiny -> 0
#
# SMD100NI stays completely untouched.
#

print("\n--- NUMERIC FIDELITY BEFORE EXPORT ---")


NUMERIC_FIDELITY_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in STRING_COLUMNS
]


fidelity_failures = []


for col in NUMERIC_FIDELITY_COLUMNS:

    expected_values = (
        pd.to_numeric(
            expected_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 28 numeric fields match "
    "the expected corrected XPT exactly."
)


# ============================================================
# 35. TEXT VALUE FIDELITY BEFORE EXPORT
# ============================================================

for col in STRING_COLUMNS:

    source_text = (
        source_df[col]
        .where(
            source_df[col].notna(),
            ""
        )
        .astype(str)
    )


    current_text = (
        df[col]
        .where(
            df[col].notna(),
            ""
        )
        .astype(str)
    )


    if not np.array_equal(
        source_text.to_numpy(),
        current_text.to_numpy()
    ):

        raise ValueError(
            f"{col} text changed before export."
        )


print(
    "PASS: SMDUPCA and SMD100BR text "
    "remain exactly unchanged."
)


# ============================================================
# 36. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO rounding.
#
# NO:
#
#     float_format=
#
# This is required because SMD100NI must retain
# its natural decimal representation.
#
# SMDUPCA remains text, preserving leading zeros.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 37. RE-READ NUMERIC DATA WITH ROUND-TRIP PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 38. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY ---"
)


csv_numeric_failures = []


for col in NUMERIC_FIDELITY_COLUMNS:

    expected_values = (
        pd.to_numeric(
            expected_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_numeric_failures.append(
            col
        )


if csv_numeric_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_numeric_failures)
    )


print(
    "PASS: All 28 numeric fields round-trip "
    "to exact corrected XPT values."
)


# ============================================================
# 39. NUMERIC MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in NUMERIC_FIDELITY_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Numeric missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Numeric missing-value positions changed."
    )


print(
    "PASS: Numeric missing positions "
    "preserved exactly."
)


# ============================================================
# 40. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 41. EXACT TEXT FIELD FIDELITY AFTER EXPORT
# ============================================================

print("\n--- TEXT FIELD FIDELITY ---")


for col in STRING_COLUMNS:

    expected_text = (
        source_df[col]
        .where(
            source_df[col].notna(),
            ""
        )
        .astype(str)
        .to_numpy()
    )


    exported_text = (
        raw_csv_df[col]
        .to_numpy()
    )


    difference_count = int(
        np.sum(
            expected_text
            !=
            exported_text
        )
    )


    print(
        f"{col} text differences:",
        difference_count
    )


    if difference_count != 0:

        raise ValueError(
            f"{col} changed during CSV export."
        )


print(
    "PASS: UPC and brand text preserved exactly."
)


# ============================================================
# 42. FINAL UPC LEADING-ZERO CHECK
# ============================================================

final_upc = (
    raw_csv_df["SMDUPCA"]
)


final_upc_nonblank = (
    final_upc != ""
)


final_upc_recorded = int(
    final_upc_nonblank.sum()
)


final_upc_12_char = int(
    final_upc[
        final_upc_nonblank
    ]
    .str.len()
    .eq(12)
    .sum()
)


final_upc_leading_zero = int(
    final_upc[
        final_upc_nonblank
    ]
    .str.startswith("0")
    .sum()
)


print("\n--- FINAL UPC CHECK ---")


print(
    "Recorded:",
    final_upc_recorded
)


print(
    "12-character:",
    final_upc_12_char
)


print(
    "Beginning with zero:",
    final_upc_leading_zero
)


if not (
    final_upc_recorded == 672
    and
    final_upc_12_char == 672
    and
    final_upc_leading_zero == 585
):

    raise ValueError(
        "SMDUPCA formatting changed."
    )


print(
    "PASS: SMDUPCA leading zeros preserved."
)


# ============================================================
# 43. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in NUMERIC_FIDELITY_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 44. .0 CHECK - WHOLE-NUMBER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT perform this check globally.
#
# SMD100NI is a genuine decimal field.
#
# A released nicotine value such as:
#
#     1.0
#
# is legitimate and should stay 1.0.
#
# Check ".0" ONLY in the 27 proven integer fields.
#

print("\n--- INTEGER COLUMN .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = (
        raw_csv_df[col]
    )


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 in "
    "the 27 whole-number fields."
)


# ============================================================
# 45. SMD100NI .0 VALUES ARE LEGITIMATE
# ============================================================

#
# Informational only.
#
# The attached file contains 58 nicotine records
# with the genuine value 1.0.
#
# These MUST NOT be changed to an integer field.
#

nicotine_dot_zero_count = int(
    (
        raw_csv_df["SMD100NI"]
        .ne("")
        &
        raw_csv_df["SMD100NI"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\nSMD100NI values ending in legitimate '.0':",
    nicotine_dot_zero_count
)


if nicotine_dot_zero_count != 58:

    raise ValueError(
        "Unexpected SMD100NI .0 representation count."
    )


print(
    "PASS: Genuine SMD100NI decimal representation "
    "was preserved."
)


# ============================================================
# 46. EXACT SMD100NI DECIMAL FIDELITY
# ============================================================

expected_nicotine = (
    expected_df["SMD100NI"]
    .to_numpy(
        dtype=float
    )
)


exported_nicotine = (
    roundtrip_df["SMD100NI"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_nicotine,
    exported_nicotine,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "SMD100NI decimal fidelity failed."
    )


print(
    "PASS: SMD100NI values preserved exactly."
)


# ============================================================
# 47. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in NUMERIC_FIDELITY_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values.eq(0).sum()
    )


    if count > 0:

        csv_zero_counts[col] = count


print("\n--- FINAL CSV ZERO PATTERN ---")


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly "
    "matches validated corrections."
)


# ============================================================
# 48. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 49. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 50. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "SMQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "SMD030 zero restorations:",
    EXPECTED_TINY_COUNTS["SMD030"]
)


print(
    "SMD641 zero restorations:",
    EXPECTED_TINY_COUNTS["SMD641"]
)


print(
    "SMD100FL zero restorations:",
    EXPECTED_TINY_COUNTS["SMD100FL"]
)


print(
    "SMD100MN zero restorations:",
    EXPECTED_TINY_COUNTS["SMD100MN"]
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "SMD100NI fractional observations:",
    nicotine_fractional_count
)


print(
    "SMD100NI legitimate whole-looking decimals:",
    nicotine_whole_count
)


print(
    "String variables:",
    STRING_COLUMNS
)


print(
    "UPC values preserved:",
    final_upc_recorded
)


print(
    "UPC values beginning with zero:",
    final_upc_leading_zero
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Text differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0 in integer fields: "
    "0 expected"
)


print(
    "SMD100NI decimal values altered: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 SMQ_G XPT -> CSV
Rows: 6,790
Columns: 30
PASS: Structure = 6,790 rows x 30 columns.

--- FIELD TYPE PLAN ---
Decimal fields: ['SMD100NI']
String fields: ['SMDUPCA', 'SMD100BR']
Whole-number fields: 27
PASS: UPC and cigarette-brand fields loaded as text/object.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 812

Tiny values by variable:
SMD100MN    652
SMD641       89
SMD030       58
SMD100FL     13

PASS: Exact 812-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
SMD030: 58 restored to 0
SMD641: 89 restored to 0
SMD100FL: 13 restored to 0
SMD100MN: 652 restored to 0

PASS: 812 validated zero values restored.

Tiny values remaining after correction: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL CHECK ---
SMD100NI: 767 fractional observations

PASS: SMD100NI is the only genuine decimal variable.

--- SMD100NI ---
Non-missing values: 825
Fractional observations: 767
Whole-looking deci